# Laboratorio: lectura e escritura con PySpark

Seguiremos os pasos de lectura, limpeza, consulta e escritura da tarefa Spark, empregando **datos meteorolóxicos** en lugar de Penguins. Practicaremos CSV, JSON, Parquet, ORC e texto en HDFS, e a escritura nunha táboa PostgreSQL mediante JDBC.

O ficheiro [meteo-lab.csv](../data/meteo-lab.csv) contén 12 rexistros de exemplo con catro columnas: `data`, `cidade`, `temperatura` e `humidade`. Hai valores ausentes para practicar o filtrado. Executa as celas en orde nun notebook de Python 3 de JupyterLab do clúster.

## Preparación

Desde PowerShell, no cartafol `big-data-cluster`, inicia o clúster base e PostgreSQL:

```powershell
docker compose -f compose.base.yml -f compose.postgres.yml up -d
docker compose -f compose.base.yml -f compose.postgres.yml ps
```

Agarda a que `postgres-serving` apareza como **healthy**. As imaxes deben incluír o driver JDBC PostgreSQL. Abre JupyterLab en `http://localhost:8888` ou no porto configurado.

Copia `data/meteo-lab.csv` destes materiais ao cartafol `spark-notebooks/data` do host. Desde o terminal de JupyterLab, sube o ficheiro a HDFS, substituíndo `teunome` polo teu identificador:

```bash
hdfs dfs -mkdir -p /tmp/teunome/meteo/datos
hdfs dfs -put /home/hadoop/work/data/meteo-lab.csv /tmp/teunome/meteo/datos/
hdfs dfs -ls /tmp/teunome/meteo/datos
```

Se xa está subido, comproba o listado antes de continuar. Os destinos de saída serán diferentes da ruta do CSV orixinal.

## 1. Lectura e esquema

Crea a sesión e define rutas fixas propias. Substitúe tamén `teunome` no código Python.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("lab-meteo-lectura-escritura")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

In [ ]:


usuario = "teunome"
base = f"/tmp/{usuario}/meteo"
entrada = f"{base}/datos/meteo-lab.csv"
ruta_csv = f"{base}/limpos_csv"
ruta_json = f"{base}/limpos_json"
ruta_parquet = f"{base}/limpos_parquet"
ruta_orc = f"{base}/limpos_orc"
ruta_texto = f"{base}/temperaturas_texto"


print("Entrada HDFS:", entrada)


Le o CSV con cabeceira e inferencia de tipos. `count()` devolve o número de filas e `printSchema()` mostra os tipos inferidos.


In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("inferSchema", True)
    .csv(entrada)
)
df.show()
df.printSchema()
n_entrada = df.count()
print("Filas de entrada:", n_entrada)
assert n_entrada == 12


## 2. Calidade dos datos

Rexistra unha vista temporal. A diferenza entre `COUNT(*)` e `COUNT(columna)` dá o número de nulos de cada columna.


In [ ]:
df.createOrReplaceTempView("meteo")
spark.sql("""
    SELECT COUNT(*) - COUNT(data) AS nulos_data,
           COUNT(*) - COUNT(cidade) AS nulos_cidade,
           COUNT(*) - COUNT(temperatura) AS nulos_temperatura,
           COUNT(*) - COUNT(humidade) AS nulos_humidade
    FROM meteo
""").show()


Conserva só as filas con data, cidade e temperatura informadas. A humidade é opcional: unha fila non se elimina só por ter ese campo nulo.


In [ ]:
df_limpo = spark.sql("""
    SELECT *
    FROM meteo
    WHERE data IS NOT NULL
      AND cidade IS NOT NULL
      AND temperatura IS NOT NULL
""")
n_limpos = df_limpo.count()
print("Filas antes:", n_entrada)
print("Filas despois:", n_limpos)
assert n_limpos == 10
df_limpo.createOrReplaceTempView("meteo_limpa")


## 3. Consulta

Calcula o número de rexistros e a temperatura media por cidade a partir dos datos limpos. O resultado é outro DataFrame.


In [ ]:
resumo = spark.sql("""
    SELECT cidade,
           COUNT(*) AS numero_rexistros,
           AVG(temperatura) AS temperatura_media
    FROM meteo_limpa
    GROUP BY cidade
    ORDER BY cidade
""")
resumo.show()


## 4. Escritura e lectura en diferentes formatos

Gardaremos os **10 rexistros limpos**, non o resumo por cidade. Usamos `errorifexists` para detectar destinos existentes. Para repetir unha escritura, podes cambiar o modo a `overwrite` unicamente sobre as túas rutas de saída.

### 4.1. Parquet

Parquet conserva o esquema, polo que non precisamos indicar os tipos ao recuperalo.


In [ ]:
df_limpo.write.mode("errorifexists").parquet(ruta_parquet)
df_parquet = spark.read.parquet(ruta_parquet)
df_parquet.show(5)
df_parquet.printSchema()
print("Filas Parquet:", df_parquet.count())
assert df_parquet.count() == n_limpos


### 4.2. CSV

Escribe con cabeceira e **punto e coma** como separador. CSV non conserva os tipos; na lectura reutilizamos o esquema dos datos limpos.


In [ ]:
(
    df_limpo.write
    .mode("errorifexists")
    .option("header", True)
    .option("sep", ";")
    .csv(ruta_csv)
)
df_csv = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .schema(df_limpo.schema)
    .csv(ruta_csv)
)
df_csv.show(5)
df_csv.printSchema()
assert df_csv.count() == n_limpos


### 4.3. JSON

Spark escribe un obxecto por liña. `ignoreNullFields=False` conserva os campos con valor nulo. Indicamos o esquema na lectura para manter os tipos.


In [ ]:
(
    df_limpo.write
    .mode("errorifexists")
    .option("ignoreNullFields", False)
    .json(ruta_json)
)
df_json = spark.read.schema(df_limpo.schema).json(ruta_json)
df_json.show(5)
df_json.printSchema()
assert df_json.count() == n_limpos


### 4.4. ORC

ORC tamén conserva o esquema no almacenamento.


In [ ]:
df_limpo.write.mode("errorifexists").orc(ruta_orc)
df_orc = spark.read.orc(ruta_orc)
df_orc.show(5)
df_orc.printSchema()
assert df_orc.count() == n_limpos


### 4.5. Texto

A escritura de texto require unha única columna de cadeas. Neste exemplo exportamos **só as temperaturas**, convertidas a texto; os outros formatos gardan as catro columnas.


In [ ]:
temperaturas = spark.sql("""
    SELECT CAST(temperatura AS STRING) AS value
    FROM meteo_limpa
""")
temperaturas.write.mode("errorifexists").text(ruta_texto)
df_texto = spark.read.text(ruta_texto)
df_texto.show(5)
df_texto.printSchema()
assert df_texto.count() == n_limpos


## 5. Ficheiros e esquemas

Executa os listados desde o notebook co cliente HDFS do contedor:


In [ ]:
import subprocess

for ruta in [ruta_parquet, ruta_csv, ruta_json, ruta_orc, ruta_texto]:
    print(f"Listado de {ruta}")
    subprocess.run(["hdfs", "dfs", "-ls", ruta], check=True)


Os destinos son directorios con ficheiros `part-...` e posibles marcadores como `_SUCCESS`. Spark escribe as particións en paralelo e pode xerar varios ficheiros. Para ler o conxunto completo indicamos o directorio, como fixemos nos exemplos anteriores.

Mostra seguidos os esquemas do CSV de entrada e do Parquet recuperado:


In [ ]:
print("CSV de entrada, con inferencia:")
df.printSchema()
print("Parquet recuperado:")
df_parquet.printSchema()


## 6. Escritura e lectura JDBC

Emprega unha táboa propia con nome fixo no esquema `public`. A conexión usa o nome do servizo e o porto interno de PostgreSQL. O driver JDBC xa debe estar dispoñible na imaxe do notebook e dos executores.


In [ ]:
jdbc_url = "jdbc:postgresql://postgres-serving:5432/gold_serving"
propiedades_jdbc = {
    "user": "gold",
    "password": "gold",
    "driver": "org.postgresql.Driver",
}
taboa_pg = f"public.meteo_{usuario}"

(
    df_limpo.write
    .mode("errorifexists")
    .option("numPartitions", "1")
    .option("batchsize", "500")
    .jdbc(url=jdbc_url, table=taboa_pg, properties=propiedades_jdbc)
)
print("Táboa PostgreSQL:", taboa_pg)


Recupera a táboa, mostra o esquema e cinco filas, e comproba o reconto dos datos limpos:


In [ ]:
df_pg = spark.read.jdbc(
    url=jdbc_url,
    table=taboa_pg,
    properties=propiedades_jdbc,
)
df_pg.printSchema()
df_pg.show(5)
print("Filas PostgreSQL:", df_pg.count())
assert df_pg.count() == n_limpos


Para repetir a carga podes usar `overwrite` unicamente sobre a túa táboa de prácticas: pode eliminala e recreala. `append` engade rexistros e repetir a mesma carga pode duplicalos.

Cando remates todas as comprobacións, pecha a sesión. Os datos escritos permanecen en HDFS e PostgreSQL.


In [ ]:
spark.stop()
